# MayFly labels on Colab

Deep labels for MayFly from a free Colab session, the Colab slot of `kaggle_label.py`. The training machine keeps a few
batches waiting in a private Kaggle dataset, `<KAGGLE_USER>/mayfly-batches`. This notebook works through them one after
another, about an hour each, and sends each result to `<KAGGLE_USER>/mayfly-colab-results`, where the training machine reads
it in within five minutes.

**Once**, set `KAGGLE_USER` in the code cell to the Kaggle user name that owns the datasets. Open Secrets (the key
icon on the left), add a secret named `KAGGLE_API_TOKEN` (or `KAGGLEAPITOKEN`) holding
a Kaggle API token, and switch on its notebook access.

**Each time**, Runtime > Change runtime type > **v5e-1 TPU** (the TPU itself is never used, but its host has 24 cores),
then Runtime > **Run all**, and keep the tab open. It stops taking batches by itself before Colab's twelve hours.
Stopping it early costs only the batch in progress, which the next session does again.

In [ ]:
# MayFly deep labels on a free Colab session. Fetches colab_worker.py from the private dataset and runs it.
import os
import signal
import subprocess
import sys
import time

START = time.time()
KAGGLE_USER = os.environ.get("KAGGLE_USER", "")   # set to the Kaggle user name that owns the datasets
DATASET = f"{KAGGLE_USER}/mayfly-batches"
ROOT = "/content/mayfly"
PY = os.path.join(ROOT, "py")                # the Kaggle CLI, kept apart from Colab's own packages

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--target", PY, "kaggle==2.2.4"],
               check=True)

from google.colab import userdata
token, why = None, ""
for name in ("KAGGLE_API_TOKEN", "KAGGLEAPITOKEN"):  # Colab may save the name without the underscores
    try:
        token = userdata.get(name)
        break
    except Exception as e:
        why = type(e).__name__
if not token:                                # raised out here, so Colab shows the message and nothing chained
    raise SystemExit("Add a Kaggle API token to Secrets as KAGGLE_API_TOKEN and switch on its notebook access "
                     f"({why}).")
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
path = os.path.expanduser("~/.kaggle/access_token")
with open(path, "w") as fh:
    fh.write(token.strip())
os.chmod(path, 0o600)
del token

env = dict(os.environ, PYTHONPATH=os.pathsep.join(p for p in (PY, os.environ.get("PYTHONPATH", "")) if p),
           MAYFLY_START=str(START), MAYFLY_ROOT=ROOT, PYTHONUNBUFFERED="1", KAGGLE_USER=KAGGLE_USER,
           PIP_DISABLE_PIP_VERSION_CHECK="1")
env.pop("KAGGLE_API_TOKEN", None)            # the token file, never a stale variable
boot = os.path.join(ROOT, "boot")
r = subprocess.run([sys.executable, "-m", "kaggle", "datasets", "download", DATASET, "-p", boot, "--unzip", "-o",
                    "-q"], env=env, capture_output=True, text=True)
worker = os.path.join(boot, "colab_worker.py")
if not os.path.exists(worker):
    raise SystemExit(f"Could not fetch {DATASET}: {(r.stdout + r.stderr).strip()[-400:]}")

p = subprocess.Popen([sys.executable, worker], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                     start_new_session=True)
try:
    for line in p.stdout:
        print(line, end="", flush=True)
    code = p.wait()
except BaseException:                        # stopped by hand, so stop the worker, its engines and all
    os.killpg(p.pid, signal.SIGTERM)
    raise
print("worker exited with code", code)